# Price Prediction RL Trading Model Training

This notebook will guide you through training a Reinforcement Learning (RL) agent for algorithmic trading using Stable Baselines 3. It includes:
- Downloading real market data using `yfinance`
- A custom OpenAI Gym environment tailored for trading
- Integration with Weights & Biases (W&B) for live logging and tracking of your model's accuracy, rewards, and performance
- Training a Proximal Policy Optimization (PPO) agent

In [ ]:
# 1. Install required dependencies
!pip install gymnasium stable-baselines3[extra] wandb pandas numpy yfinance tensorboard

In [ ]:
import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.callbacks import BaseCallback
import numpy as np
import pandas as pd
import yfinance as yf
import wandb
from wandb.integration.sb3 import WandbCallback

In [ ]:
# 2. Log in to Weights & Biases (W&B)
# This will prompt you for an API key. You can get one for free by signing up at https://wandb.ai
wandb.login()

### Defining the Custom Trading Environment
This environment keeps track of our balance, the shares we hold, and calculates rewards based on the **net worth** (Cash + Shares * Current Price). I have added transaction fees to make the agent's behavior more realistic and accurate.

In [ ]:
class TradingEnv(gym.Env):
    def __init__(self, data, initial_balance=10000, transaction_fee_percent=0.001):
        super(TradingEnv, self).__init__()
        self.data = data.reset_index(drop=True)
        self.initial_balance = initial_balance
        self.transaction_fee_percent = transaction_fee_percent
        
        # Action space: 0 = Hold, 1 = Buy, 2 = Sell
        self.action_space = spaces.Discrete(3)
        
        # Observation space: [Current Price, Volume, Balance, Shares Held, Net Worth]
        self.observation_space = spaces.Box(
            low=-np.inf, high=np.inf, shape=(5,), dtype=np.float32
        )
        
    def reset(self, seed=None):
        super().reset(seed=seed)
        self.current_step = 0
        self.balance = self.initial_balance
        self.shares_held = 0
        self.net_worth = self.initial_balance
        
        return self._get_observation(), {}
        
    def _get_observation(self):
        obs = np.array([
            self.data.loc[self.current_step, 'Close'],
            self.data.loc[self.current_step, 'Volume'],
            self.balance,
            self.shares_held,
            self.net_worth
        ], dtype=np.float32)
        return obs
        
    def step(self, action):
        current_price = self.data.loc[self.current_step, 'Close']
        prev_net_worth = self.net_worth
        
        # Execute Action
        if action == 1: # Buy
            # Buy as many shares as possible with current balance
            shares_to_buy = self.balance // (current_price * (1 + self.transaction_fee_percent))
            if shares_to_buy > 0:
                cost = shares_to_buy * current_price * (1 + self.transaction_fee_percent)
                self.balance -= cost
                self.shares_held += shares_to_buy
                
        elif action == 2: # Sell
            # Sell all held shares
            if self.shares_held > 0:
                revenue = self.shares_held * current_price * (1 - self.transaction_fee_percent)
                self.balance += revenue
                self.shares_held = 0
                
        # Move to next step
        self.current_step += 1
        
        # Check if done
        done = self.net_worth <= 0 or self.current_step >= len(self.data) - 1
        
        # Update Net Worth
        if not done:
            new_price = self.data.loc[self.current_step, 'Close']
        else:
            new_price = current_price
            
        self.net_worth = self.balance + self.shares_held * new_price
        
        # Reward is the change in net worth (profit/loss)
        reward = self.net_worth - prev_net_worth
        
        info = {
            'step': self.current_step,
            'net_worth': self.net_worth,
            'balance': self.balance,
            'shares': self.shares_held
        }
        
        return self._get_observation(), reward, done, False, info

In [ ]:
# 3. Download Real Market Data
# You can change 'AAPL' to any ticker like 'TSLA', 'BTC-USD', etc.
print("Downloading stock data...")
df = yf.download('AAPL', start='2015-01-01', end='2023-12-31')
df = df.dropna()

# Flatten MultiIndex columns (Fix for newer yfinance versions)
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

df.head()

### Training the Agent
We use PPO (Proximal Policy Optimization). The `WandbCallback` hooks into the training process to log metrics automatically to your Weights & Biases dashboard.

In [ ]:
# 4. Initialize W&B Run
run = wandb.init(
    project="rl-trading-bot",
    sync_tensorboard=True,  # Sync metrics to W&B
    name="PPO-AAPL-Training"
)

# 5. Create Vectorized Environment
env = make_vec_env(lambda: TradingEnv(df), n_envs=4)

# 6. Initialize PPO Model
model = PPO(
    "MlpPolicy",
    env,
    verbose=1,
    tensorboard_log=f"runs/{run.id}",
    learning_rate=0.0003,
    n_steps=2048,
    batch_size=64,
    ent_coef=0.01, # Encourage exploration
)

In [ ]:
# 7. Train the Model
print("Starting training...")
model.learn(
    total_timesteps=200000, # Increase for better accuracy if needed
    callback=WandbCallback(
        gradient_save_freq=100,
        model_save_path=f"models/{run.id}",
        verbose=2,
    ),
)

# 8. Save Model and Finish Run
model.save("ppo_trading_model_final")
run.finish()
print("Training complete and model saved! Check wandb.ai for logs.")

### Testing the Model
Let's see how our trained model performs on the environment.

In [ ]:
test_env = TradingEnv(df)
obs, info = test_env.reset()

for i in range(500):
    action, _states = model.predict(obs)
    obs, reward, done, truncated, info = test_env.step(action)
    if done:
        break

print(f"Final Net Worth: ${test_env.net_worth:.2f}")